# CNN: Reconhecendo dígitos escritos à mão

**Objetivo:** mostrar como uma Rede Neural Convolucional (CNN) aprende a reconhecer imagens.

**Dataset:** MNIST, com 70 mil imagens de dígitos de 0 a 9 (28x28 pixels, tons de cinza).

**Por que CNN?** Ela extrai padrões espaciais e locais das imagens (bordas, curvas, formas) de forma automática.

## 1. Importar as bibliotecas
Usamos o TensorFlow/Keras para criar a rede e o Matplotlib para mostrar as imagens.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

## 2. Carregar e preparar os dados
- O MNIST já vem dividido em **treino** (60 mil imagens) e **teste** (10 mil imagens).
- Os pixels vão de 0 a 255. Dividimos por 255 para ficarem entre 0 e 1, o que ajuda o treino.
- Adicionamos o "canal" da imagem (1 = tons de cinza).

In [ ]:
(x_treino, y_treino), (x_teste, y_teste) = tf.keras.datasets.mnist.load_data()

x_treino = x_treino.reshape(-1, 28, 28, 1) / 255.0
x_teste = x_teste.reshape(-1, 28, 28, 1) / 255.0

print("Imagens de treino:", x_treino.shape[0])
print("Imagens de teste:", x_teste.shape[0])

# Mostra alguns exemplos
plt.figure(figsize=(8, 2))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_treino[i].reshape(28, 28), cmap="gray")
    plt.title(f"Dígito: {y_treino[i]}")
    plt.axis("off")
plt.show()

## 3. Construir a CNN
A rede é montada em camadas:

- **Conv2D:** filtros pequenos (3x3) percorrem a imagem procurando padrões, como linhas e curvas.
- **MaxPooling2D:** reduz o tamanho da imagem mantendo só o mais importante.
- **Flatten:** transforma o resultado em um vetor.
- **Dense:** camadas que tomam a decisão.
- **Softmax (saída):** dá a probabilidade de cada dígito de 0 a 9.

In [ ]:
modelo = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),   # extrai padrões simples (bordas, traços)
    layers.MaxPooling2D((2, 2)),                    # reduz a imagem
    layers.Conv2D(64, (3, 3), activation="relu"),   # extrai padrões mais complexos
    layers.MaxPooling2D((2, 2)),                    # reduz de novo
    layers.Flatten(),                               # transforma em vetor
    layers.Dense(64, activation="relu"),            # camada de decisão
    layers.Dense(10, activation="softmax")          # 10 saídas: dígitos 0 a 9
])

modelo.summary()

## 4. Treinar o modelo
- **Otimizador (adam):** ajusta os filtros da rede para errar menos.
- **Loss:** mede o quanto o modelo errou.
- **Épocas:** cada época é uma passada completa por todas as imagens de treino. Usamos 5.
- **validation_split:** separa 10% do treino para conferir se o modelo está aprendendo de verdade.

In [ ]:
modelo.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

historico = modelo.fit(x_treino, y_treino, epochs=5, validation_split=0.1)

## 5. Avaliar com imagens que o modelo nunca viu
Usamos as 10 mil imagens de teste, que ficaram de fora do treino. Assim sabemos se o modelo realmente aprendeu, e não só decorou.

In [ ]:
perda, acuracia = modelo.evaluate(x_teste, y_teste)
print(f"Acurácia no teste: {acuracia:.2%}")

## 6. Ver as previsões
Aqui o modelo prevê dígitos novos. O título mostra o que ele previu e o que era o correto.

In [ ]:
previsoes = modelo.predict(x_teste[:5])

plt.figure(figsize=(10, 2.5))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_teste[i].reshape(28, 28), cmap="gray")
    plt.title(f"Previu: {np.argmax(previsoes[i])}\nReal: {y_teste[i]}")
    plt.axis("off")
plt.show()

## Conclusão
- A CNN aprendeu a reconhecer dígitos com acurácia próxima de 99%.
- As camadas convolucionais extraem os padrões visuais automaticamente, sem precisarmos programar regras à mão.
- Esse mesmo princípio é usado em reconhecimento facial, carros autônomos e diagnóstico médico por imagem.